# Combining and checking the human annotations

Works for **any number of annotators (1, 2, 3, ...)** and for the pilot file, the main file, or partial files. Run this notebook twice:

1. **After the pilot** (`MODE = "pilot"`): shows how much the annotators agree per question and which wording is confusing, so the rubric can be fixed before the main round.
2. **After the main round** (`MODE = "main"`): validates every returned annotation file, merges them into `processed/annotations/annotations_<MODE>_raw.csv` (one row per annotator x item) and reports completeness and agreement.

**Input:** the CSV files the annotators export from their forms (`ratings_pilot_A1.csv`, `ratings_main_A1.csv`, ...), placed in `processed/annotations/incoming/`.

**Output** (in `processed/annotations/`):
| File | What it is |
|---|---|
| `annotations_<MODE>_raw.csv` | all raters stacked: one row per annotator x item. **This is the file Step 9 (LLM judges) reads.** |
| `annotations_<MODE>_consensus.csv` | one row per item: majority / median of the raters (with 1 rater, that rater's answers) |

**Highlights**
- Agreement is computed properly: **Krippendorff's alpha** (works for 2+ raters, any mix of missing items) and the **mean of pairwise Cohen's kappa** (one kappa per annotator pair, never pooled).
- With only one annotator file available, the notebook skips agreement instead of failing.
- Time statistics ignore "tab left open" outliers.
- An **applicability audit** (did annotators use "not applicable" correctly?) and a list of the **most disputed items** for the pilot meeting.

In [1]:
import sys, json, itertools
from pathlib import Path
import numpy as np, pandas as pd


def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "src" / "config.py").exists():
            return p
    raise FileNotFoundError("Open the notebook from inside the FoodJudgeBench folder.")


ROOT = find_project_root(); sys.path.insert(0, str(ROOT / "src"))
from config import PROC_DIR

MODE = "pilot"            # "pilot" or "main": which ratings_<MODE>_*.csv files to read
SECONDS_CAP = 1800        # time per item above this = tab left open; ignored in time statistics
BANK = PROC_DIR / "case_bank"
ANN = PROC_DIR / "annotations"; (ANN / "incoming").mkdir(parents=True, exist_ok=True)
spec = json.load(open(BANK / "rubric.json", encoding="utf-8"))
RUBRIC, COLS = spec["rubric"], spec["columns"]
RTYPE = {r["id"]: r["type"] for r in RUBRIC}
YN = [r["id"] for r in RUBRIC if r["type"].startswith("yn")]
SCALE = [r["id"] for r in RUBRIC if r["type"].startswith("scale")]
key = pd.read_csv(BANK / "answer_key.csv").set_index("case_id")

#### 1. Load and validate the returned files

In [2]:
files = sorted((ANN / "incoming").glob(f"ratings_{MODE}_*.csv"))
assert files, f"no ratings_{MODE}_*.csv files in {ANN/'incoming'}"
frames, issues = [], []
for f in files:
    d = pd.read_csv(f, dtype=str, keep_default_na=False)
    miss_cols = set(COLS) - set(d.columns)
    if miss_cols:
        issues.append(f"{f.name}: missing columns {sorted(miss_cols)}"); continue
    d["_file"] = f.name; frames.append(d)
raw = pd.concat(frames, ignore_index=True)
raw = raw[raw["case_id"].str.len() > 0]

if MODE == "main":
    assigned = {a: set(pd.read_csv(BANK / f"assignment_{a}.csv")["case_id"]) for a in raw["annotator_id"].unique()}
else:
    pil = set(pd.read_csv(BANK / "pilot_case_ids.csv")["case_id"])
    assigned = {a: pil for a in raw["annotator_id"].unique()}
for a, g in raw.groupby("annotator_id"):
    dup = g["case_id"].duplicated().sum()
    extra = set(g["case_id"]) - assigned[a]
    todo = assigned[a] - set(g["case_id"])
    if dup: issues.append(f"{a}: {dup} duplicate rows")
    if extra: issues.append(f"{a}: {len(extra)} items that were not assigned to this annotator")
    print(f"{a}: {len(g)} rated, {len(todo)} still missing")
cases = raw[raw["item_type"].isin(["case", "diagnostic"])]
bad = []
for r in RUBRIC:
    ok = {"yn": {"yes", "no"}, "yn_na": {"yes", "no", "na"}, "scale": set("12345"), "scale_na": set("12345") | {"na"}}[r["type"]]
    b = cases[~cases[r["id"]].isin(ok)]
    if len(b): bad.append((r["id"], len(b)))
if bad: issues.append(f"invalid / empty answers: {bad}")
print("\nISSUES:" if issues else "\nNo validation issues."); [print(" -", i) for i in issues]
N_ANN = raw["annotator_id"].nunique()
print(f"\n{N_ANN} annotator(s); raters per item: {raw.groupby('case_id')['annotator_id'].nunique().value_counts().to_dict()}")

A1: 10 rated, 0 still missing
A2: 10 rated, 0 still missing
A3: 10 rated, 0 still missing

No validation issues.

3 annotator(s); raters per item: {3: 10}


#### 2. Save the merged raw data (this is what Step 9 reads)

In [3]:
raw_out = raw.drop(columns="_file")
raw_out.to_csv(ANN / f"annotations_{MODE}_raw.csv", index=False)
print("saved", ANN / f"annotations_{MODE}_raw.csv", raw_out.shape)
sec = pd.to_numeric(raw["seconds"], errors="coerce")
ok_sec = sec[sec <= SECONDS_CAP]
print(f"\nTime per item: median {ok_sec.median():.0f}s (items over {SECONDS_CAP}s ignored: {(sec > SECONDS_CAP).sum()} of {sec.notna().sum()})")
print((raw.assign(sec=sec).query("sec <= @SECONDS_CAP").groupby("annotator_id")["sec"].median().round(0)).to_string())
print("-> the time counter runs while the tab is open, so very long values are breaks, not reading time.")

saved C:\HP Data\University-Masters\Semester_3\P1_Natural Language Processing\Lab\Project\FoodJudgeBench\processed\annotations\annotations_pilot_raw.csv (30, 19)

Time per item: median 370s (items over 1800s ignored: 2 of 30)
annotator_id
A1    424.0
A2    110.0
A3    705.0
-> the time counter runs while the tab is open, so very long values are breaks, not reading time.


#### 3. Applicability audit - did annotators use "not applicable" correctly?
Whether a question applies is a fact about the profile, not an opinion, so wrong N/A use is a wording/training problem. (Needs `case_inputs.jsonl`.)

In [4]:
ci_path = PROC_DIR / "case_inputs.jsonl"
if ci_path.exists() and cases.shape[0]:
    ci = {}
    for l in open(ci_path, encoding="utf-8"):
        if l.strip():
            c = json.loads(l); ci[c["case_key"]] = c
    kk = key.reindex(cases["case_id"])
    ap = pd.DataFrame({"case_id": cases["case_id"].values, "annotator_id": cases["annotator_id"].values})
    ap["case_key"] = kk["case_key"].values; ap["source"] = kk["profile_source"].values
    has_req = [bool(ci[k]["profile_view"]["constraints"]) or ci[k]["profile_view"]["time"]["value"] is not None if k in ci else np.nan for k in ap["case_key"]]
    has_goal = [(ci[k].get("goal_key") is not None) if k in ci else np.nan for k in ap["case_key"]]
    ap["constraint_na_expected"] = [not x if x == x else np.nan for x in has_req]
    ap["goal_na_expected"] = [not x if x == x else np.nan for x in has_goal]
    ap["certainty_na_expected"] = (ap["source"] == "synthetic").values
    res = []
    for q, col in [("ignored_constraint", "constraint_na_expected"), ("goal_alignment", "goal_na_expected"), ("overstated_certainty", "certainty_na_expected")]:
        said_na = (cases[q].values == "na"); exp = ap[col].astype(float).values
        m = ~np.isnan(exp)
        res.append({"question": q, "expected_NA_items": int(exp[m].sum()), "annotator_said_NA": int(said_na[m].sum()),
                    "NA_wrongly_used": int((said_na[m] & (exp[m] == 0)).sum()), "NA_missed": int((~said_na[m] & (exp[m] == 1)).sum()), "n": int(m.sum())})
    print(pd.DataFrame(res).to_string(index=False))
    print("\nNA_wrongly_used / NA_missed > 0 means annotators disagree with the profile facts about applicability. Clarify in the guide:")
    print(" - ignored_constraint: N/A only if the profile states NO diet, allergy AND NO time limit")
    print(" - goal_alignment: N/A only if the profile states no health goal")
    print(" - overstated_certainty: N/A only if the profile is fully stated (synthetic); every historical profile is inferred")
else:
    print("skipped (case_inputs.jsonl not found)")

            question  expected_NA_items  annotator_said_NA  NA_wrongly_used  NA_missed  n
  ignored_constraint                 12                  9                2          5 30
      goal_alignment                 24                 22                0          2 30
overstated_certainty                 12                 11                1          2 30

NA_wrongly_used / NA_missed > 0 means annotators disagree with the profile facts about applicability. Clarify in the guide:
 - ignored_constraint: N/A only if the profile states NO diet, allergy AND NO time limit
 - goal_alignment: N/A only if the profile states no health goal
 - overstated_certainty: N/A only if the profile is fully stated (synthetic); every historical profile is inferred


#### 4. Agreement between annotators (needs 2+ annotators on the same items)
- **exact agreement**: share of annotator pairs that gave the same answer;
- **alpha**: Krippendorff's alpha (nominal for yes/no/N/A; ordinal for 1-5 scales, N/A excluded and reported separately). 1 = perfect, 0 = chance. Above about 0.67 is usable, below 0.4 means the question is not reliably answerable yet;
- **kappa**: mean of Cohen's kappa over annotator pairs (quadratic-weighted for 1-5 scales);
- **yes_rate**: share of "yes". If it is below 0.1 or above 0.9, kappa and alpha are unstable (almost everyone says the same thing, so a single disagreement counts a lot) - read exact agreement instead.

In [5]:
def kripp_alpha(units, ordinal=False, order=None):
    units = [u for u in units if len(u) >= 2]
    if not units: return np.nan
    cats = order if order is not None else sorted({v for u in units for v in u})
    idx = {c: i for i, c in enumerate(cats)}; k = len(cats)
    o = np.zeros((k, k))
    for u in units:
        m = len(u)
        for a in range(m):
            for b in range(m):
                if a != b: o[idx[u[a]], idx[u[b]]] += 1 / (m - 1)
    nc = o.sum(1); n = nc.sum()
    if n <= 1: return np.nan
    if ordinal:
        d = np.zeros((k, k))
        for i in range(k):
            for j in range(k):
                if i != j:
                    lo, hi = min(i, j), max(i, j)
                    d[i, j] = (nc[lo:hi + 1].sum() - (nc[lo] + nc[hi]) / 2) ** 2
    else:
        d = 1 - np.eye(k)
    De = (np.outer(nc, nc) * d).sum() / (n * (n - 1))
    return np.nan if De == 0 else 1 - (o * d).sum() / n / De


def cohen_kappa(a, b, weights=None, cats=None):
    a, b = list(a), list(b)
    if not a: return np.nan
    cats = cats if cats is not None else sorted(set(a) | set(b)); k = len(cats); idx = {c: i for i, c in enumerate(cats)}
    O = np.zeros((k, k))
    for x, y in zip(a, b): O[idx[x], idx[y]] += 1
    O /= O.sum(); E = np.outer(O.sum(1), O.sum(0))
    i, j = np.indices((k, k))
    W = (i != j).astype(float) if weights is None else ((i - j) / max(k - 1, 1)) ** 2
    den = (W * E).sum()
    return np.nan if den == 0 else 1 - (W * O).sum() / den


def agreement_table(df):
    rows = []
    for q in YN + SCALE:
        piv = df.pivot_table(index="case_id", columns="annotator_id", values=q, aggfunc="first")
        piv = piv[piv.notna().sum(axis=1) >= 2]
        if piv.empty: continue
        isscale = q in SCALE
        if isscale:
            units = [[int(v) for v in r.dropna() if v != "na"] for _, r in piv.iterrows()]
            alpha = kripp_alpha(units, ordinal=True, order=[1, 2, 3, 4, 5])
        else:
            units = [list(r.dropna()) for _, r in piv.iterrows()]
            alpha = kripp_alpha(units)
        ex, ks, w1, nad = [], [], [], []
        for a, b in itertools.combinations(piv.columns, 2):
            s = piv[[a, b]].dropna()
            if s.empty: continue
            ex += list((s[a] == s[b]).values)
            if isscale:
                nad += list(((s[a] == "na") != (s[b] == "na")).values)
                t = s[(s[a] != "na") & (s[b] != "na")]
                if len(t):
                    w1 += list((abs(t[a].astype(int) - t[b].astype(int)) <= 1).values)
                    ks.append(cohen_kappa(t[a].astype(int), t[b].astype(int), weights="quadratic", cats=[1, 2, 3, 4, 5]))
            else:
                ks.append(cohen_kappa(s[a], s[b]))
        vals = df[q]
        row = {"question": q, "items": len(piv), "pairs": len(ex), "exact_agreement": round(float(np.mean(ex)), 2),
               "alpha": None if alpha != alpha else round(alpha, 2),
               "mean_pair_kappa": None if not ks or all(k != k for k in ks) else round(float(np.nanmean(ks)), 2)}
        row["na_rate"] = round(float((vals == "na").mean()), 2)
        if isscale:
            row["within_1_point"] = round(float(np.mean(w1)), 2) if w1 else np.nan
            row["na_disagreement"] = round(float(np.mean(nad)), 2)
        else:
            nn = vals[vals.isin(["yes", "no"])]
            row["yes_rate"] = round(float((nn == "yes").mean()), 2) if len(nn) else np.nan
        rows.append(row)
    return pd.DataFrame(rows).set_index("question")


if (raw.groupby("case_id")["annotator_id"].nunique() >= 2).any():
    agr = agreement_table(cases)
    print(agr.to_string())
    print("\nWeak spots (alpha < 0.4, or yes/no exact agreement < 0.7) - reword these questions before the next round:")
    reasons = {}
    for q, r in agr.iterrows():
        unstable = (q in YN and (r.get("yes_rate", .5) < .1 or r.get("yes_rate", .5) > .9)) or r["na_rate"] >= .8
        low = (r["alpha"] if r["alpha"] == r["alpha"] and r["alpha"] is not None else -1) < .4 or (q in YN and r["exact_agreement"] < .7)
        if low:
            reasons[q] = "alpha unstable (almost everyone gives the same answer / N/A) - check exact agreement" if unstable else "reword"
    weak = pd.Series(reasons, dtype=object)
    print(weak.to_string() if len(weak) else "none")
else:
    print("Only one annotator per item -> agreement cannot be computed. Human reliability stays unknown; say so in the report.")

                          items  pairs  exact_agreement  alpha  mean_pair_kappa  na_rate  yes_rate  within_1_point  na_disagreement
question                                                                                                                           
ignored_constraint           10     30             0.57   0.10             0.16     0.30      0.05             NaN              NaN
factual_error                10     30             0.73   0.39             0.37     0.00      0.30             NaN              NaN
invented_personalization     10     30             0.73   0.48             0.48     0.00      0.50             NaN              NaN
unsupported_claim            10     30             0.67   0.28             0.27     0.00      0.33             NaN              NaN
overstated_certainty         10     30             0.57   0.34             0.32     0.37      0.32             NaN              NaN
clarity                      10     30             0.33   0.29             0

#### 5. Most disputed items (for the pilot meeting)

In [6]:
if (raw.groupby("case_id")["annotator_id"].nunique() >= 2).any():
    dis = []
    for cid, g in cases.groupby("case_id"):
        if len(g) < 2: continue
        n_dis = 0
        for q in YN + SCALE:
            v = g[q].tolist()
            n_dis += (len(set(v)) > 1) if q in YN else (max(int(x) for x in v if x != "na") - min(int(x) for x in v if x != "na") >= 2 if any(x != "na" for x in v) else 0) or (len(set(x == "na" for x in v)) > 1)
        dis.append({"case_id": cid, "questions_in_dispute": int(n_dis), "comments": " | ".join(c for c in g["comment"] if c)[:200]})
    print(pd.DataFrame(dis).sort_values("questions_in_dispute", ascending=False).head(6).to_string(index=False))
    print("(dispute = yes/no differ, or scores differ by 2+ points, or one says N/A and another does not)")

case_id  questions_in_dispute comments
C-A65BE                     7         
C-37157                     5         
C-CF2B8                     5         
C-A762B                     5         
C-C9CF9                     5         
C-1102C                     4         
(dispute = yes/no differ, or scores differ by 2+ points, or one says N/A and another does not)


#### 6. Do annotators catch the injected errors?

In [7]:
FLAG = {"invented_preference": "invented_personalization", "wrong_nutrition": "factual_error",
        "ignored_time_limit": "ignored_constraint", "ignored_dietary_constraint": "ignored_constraint"}
dg = cases[cases["item_type"] == "diagnostic"].join(key[["injected_error"]], on="case_id")
if len(dg):
    dg["caught"] = [r[FLAG[r["injected_error"]]] == "yes" for _, r in dg.iterrows()]
    print(dg.groupby("injected_error")["caught"].agg(["mean", "count"]).round(2).rename(columns={"mean": "share_of_ratings_that_flag_it", "count": "ratings"}))
    missing = sorted(set(FLAG) - set(dg["injected_error"]))
    if missing: print("\nNOT covered by these items:", missing, "(only the types above were tested)")
else:
    print("no diagnostic items in this file")

                     share_of_ratings_that_flag_it  ratings
injected_error                                             
invented_preference                           0.83        6

NOT covered by these items: ['ignored_dietary_constraint', 'ignored_time_limit', 'wrong_nutrition'] (only the types above were tested)


#### 7. Pairwise items

In [8]:
pr = raw[raw["item_type"] == "pair"]
if len(pr):
    pr = pr.join(key[["A", "B"]], on="case_id")
    pr["winner"] = np.where(pr["pair_choice"] == "A", pr["A"], np.where(pr["pair_choice"] == "B", pr["B"], "tie"))
    print(pr["winner"].value_counts().to_string())
else:
    print("no pairwise items in this file")

no pairwise items in this file


#### 8. Consensus file (one row per item)
yes/no: majority (a tie is kept as `tie`). 1-5: median of the numeric answers. "na" wins when more than half of the raters said N/A.
With a single annotator the consensus is simply that annotator's answers (and `n_raters` = 1, so later results are marked as single-rater).

In [11]:
def consensus(g, q):
    v = [x for x in g[q] if x != ""]
    if not v: return ""
    if RTYPE[q] in ("yn", "yn_na"):
        c = pd.Series(v).value_counts()
        return c.index[0] if len(c) == 1 or c.iloc[0] > c.iloc[1] else "tie"
    na = sum(x == "na" for x in v)
    if na * 2 > len(v): return "na"
    nums = [int(x) for x in v if x != "na"]
    return float(np.median(nums)) if nums else "na"


rows = []
for cid, g in raw.groupby("case_id"):
    it = g["item_type"].iloc[0]
    r = {"case_id": cid, "item_type": it, "n_raters": g["annotator_id"].nunique()}
    if it == "pair":
        c = g["pair_choice"].value_counts()
        r["pair_choice"] = c.index[0] if len(c) == 1 or c.iloc[0] > c.iloc[1] else "tie"
    else:
        for q in YN + SCALE: r[q] = consensus(g, q)
    rows.append(r)
cons = pd.DataFrame(rows)
cons.to_csv(ANN / f"annotations_{MODE}_consensus.csv", index=False)
print("saved", ANN / f"annotations_{MODE}_consensus.csv", cons.shape)
print("items with a yes/no tie:", int((cons[YN] == "tie").any(axis=1).sum()) if len(cons) else 0)
print(f"\nNext: Notebook 9 (LLM_Judge) reads annotations_{MODE}_raw.csv (set HUMAN_FILES in 9_LLM_Judge.ipynb).")

saved C:\HP Data\University-Masters\Semester_3\P1_Natural Language Processing\Lab\Project\FoodJudgeBench\processed\annotations\annotations_pilot_consensus.csv (10, 13)
items with a yes/no tie: 2

Next: Notebook 9 (LLM_Judge) reads annotations_pilot_raw.csv (set HUMAN_FILES in 9_LLM_Judge.ipynb).
